In [2]:
%pip install nba_api

In [11]:
!pip -q install pandas requests tqdm

In [12]:
import requests
import pandas as pd
import time
from pathlib import Path
from tqdm.auto import tqdm
from datetime import date

# ==========================================
# CONFIGURATION
# ==========================================

BASE_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Dataset/NBA Dataset"
)

RAW_DIR = BASE_DIR / "raw"
CLEAN_DIR = BASE_DIR / "clean"

RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

API_URL = (
    "https://site.api.espn.com/apis/site/v2/"
    "sports/basketball/nba/scoreboard"
)

# 2015-16 through 2024-25
SEASONS = range(2015, 2025)

# Only these team names represent the NBA's
# 30 franchises during the selected period.
NBA_TEAMS = {
    "Atlanta Hawks",
    "Boston Celtics",
    "Brooklyn Nets",
    "Charlotte Hornets",
    "Chicago Bulls",
    "Cleveland Cavaliers",
    "Dallas Mavericks",
    "Denver Nuggets",
    "Detroit Pistons",
    "Golden State Warriors",
    "Houston Rockets",
    "Indiana Pacers",
    "Los Angeles Clippers",
    "LA Clippers",
    "Los Angeles Lakers",
    "Memphis Grizzlies",
    "Miami Heat",
    "Milwaukee Bucks",
    "Minnesota Timberwolves",
    "New Orleans Pelicans",
    "New York Knicks",
    "Oklahoma City Thunder",
    "Orlando Magic",
    "Philadelphia 76ers",
    "Phoenix Suns",
    "Portland Trail Blazers",
    "Sacramento Kings",
    "San Antonio Spurs",
    "Toronto Raptors",
    "Utah Jazz",
    "Washington Wizards"
}

# NBA Cup championship games are not
# part of the official regular-season record.
NBA_CUP_FINALS = {
    "2023-24": "2023-12-09",
    "2024-25": "2024-12-17"
}

# Season-specific expected game counts.
# 2019-20 was interrupted by COVID-19.
# 2020-21 had a 72-game schedule.
# Earlier seasons have 1230 games.
EXPECTED = {
    "2015-16": 1230,
    "2016-17": 1230,
    "2017-18": 1230,
    "2018-19": 1230,
    "2019-20": None,  # Pandemic-affected
    "2020-21": 1080,
    "2021-22": 1230,
    "2022-23": 1230,
    "2023-24": 1230,
    "2024-25": 1230
}

session = requests.Session()

def fetch_day(day, retries=3):
    date_str = day.strftime("%Y%m%d")

    for attempt in range(retries):
        try:
            response = session.get(
                API_URL,
                params={
                    "dates": date_str,
                    "limit": 100
                },
                timeout=30
            )
            response.raise_for_status()
            return response.json().get("events", [])

        except requests.RequestException as e:
            print(f"  {date_str}: {e}")

            if attempt < retries - 1:
                time.sleep(3 * (attempt + 1))

    return None


def extract_game(event, season):
    try:
        if not event.get("status", {}).get(
            "type", {}
        ).get("completed", False):
            return None

        if event.get("season", {}).get("type") != 2:
            return None

        competition = event["competitions"][0]
        competitors = competition["competitors"]

        home = next(
            x for x in competitors
            if x["homeAway"] == "home"
        )
        away = next(
            x for x in competitors
            if x["homeAway"] == "away"
        )

        return {
            "SEASON": season,
            "GAME_ID": str(event["id"]),
            "GAME_DATE": event["date"],
            "HOME_TEAM_ID": home["team"]["id"],
            "HOME_TEAM": home["team"]["displayName"],
            "AWAY_TEAM_ID": away["team"]["id"],
            "AWAY_TEAM": away["team"]["displayName"],
            "HOME_SCORE": int(home["score"]),
            "AWAY_SCORE": int(away["score"]),
            "HOME_WIN": int(
                int(home["score"]) > int(away["score"])
            ),
            "EVENT_NAME": event.get("name", "")
        }

    except (KeyError, ValueError, StopIteration, TypeError):
        return None


def clean_games(df, season):
    clean = df.copy()

    # Keep games involving NBA teams only
    clean = clean[
        clean["HOME_TEAM"].isin(NBA_TEAMS) &
        clean["AWAY_TEAM"].isin(NBA_TEAMS)
    ].copy()

    # Normalize Clippers name
    clean["HOME_TEAM"] = clean["HOME_TEAM"].replace(
        {"Los Angeles Clippers": "LA Clippers"}
    )
    clean["AWAY_TEAM"] = clean["AWAY_TEAM"].replace(
        {"Los Angeles Clippers": "LA Clippers"}
    )

    # Remove the NBA Cup championship.
    # Use US Eastern date to avoid UTC day shifts.
    if season in NBA_CUP_FINALS:
        local_date = pd.to_datetime(
            clean["GAME_DATE"], utc=True
        ).dt.tz_convert(
            "America/New_York"
        ).dt.strftime("%Y-%m-%d")

        final_date = NBA_CUP_FINALS[season]

        cup_final = (
            (local_date == final_date) &
            (
                (
                    clean["HOME_TEAM"].isin(
                        ["Milwaukee Bucks",
                         "Oklahoma City Thunder"]
                    )
                    &
                    clean["AWAY_TEAM"].isin(
                        ["Milwaukee Bucks",
                         "Oklahoma City Thunder"]
                    )
                )
                if season == "2024-25"
                else (
                    clean["HOME_TEAM"].isin(
                        ["Los Angeles Lakers",
                         "Indiana Pacers"]
                    )
                    &
                    clean["AWAY_TEAM"].isin(
                        ["Los Angeles Lakers",
                         "Indiana Pacers"]
                    )
                )
            )
        )

        clean = clean.loc[~cup_final].copy()

    clean = clean.drop_duplicates(
        subset=["GAME_ID"]
    )

    return clean.sort_values("GAME_DATE")


def validate(df, season):
    home = df.groupby("HOME_TEAM").size()
    away = df.groupby("AWAY_TEAM").size()

    counts = home.add(
        away, fill_value=0
    ).astype(int)

    expected = EXPECTED[season]

    print(f"\nSeason: {season}")
    print("Total games:", len(df))
    print("Unique teams:", len(counts))

    if expected is not None:
        print("Expected:", expected)

        if len(df) != expected:
            print("WARNING: Game count mismatch")

    if season not in ("2019-20", "2020-21"):
        incorrect = counts[counts != 82]

        if not incorrect.empty:
            print("Teams without 82 games:")
            print(incorrect.to_string())

    if season == "2020-21":
        incorrect = counts[counts != 72]

        if not incorrect.empty:
            print("Teams without 72 games:")
            print(incorrect.to_string())

    return counts


# ==========================================
# MAIN DOWNLOAD
# ==========================================

all_clean = []
validation_results = []

for start_year in SEASONS:

    season = (
        f"{start_year}-"
        f"{str(start_year + 1)[-2:]}"
    )

    raw_path = RAW_DIR / f"nba_games_{season}_raw.csv"
    clean_path = CLEAN_DIR / f"nba_games_{season}_clean.csv"

    print(f"\n{'=' * 50}")
    print("Processing:", season)

    # Reuse downloaded raw files if present
    if raw_path.exists():
        print("Loading existing raw dataset...")
        raw = pd.read_csv(raw_path)

    else:
        # Extra days around the regular-season
        # window, including pandemic restart.
        start = pd.Timestamp(f"{start_year}-10-01")

        if start_year == 2019:
            end = pd.Timestamp("2020-08-31")
        elif start_year == 2020:
            start = pd.Timestamp("2020-12-01")
            end = pd.Timestamp("2021-05-31")
        else:
            end = pd.Timestamp(
                f"{start_year + 1}-04-30"
            )

        dates = pd.date_range(start, end)

        records = []
        failed_dates = []

        for day in tqdm(dates, desc=season):
            events = fetch_day(day)

            if events is None:
                failed_dates.append(
                    day.strftime("%Y%m%d")
                )
                continue

            for event in events:
                record = extract_game(event, season)

                if record is not None:
                    records.append(record)

            time.sleep(0.25)

        if not records:
            print("No games downloaded.")
            continue

        raw = pd.DataFrame(records).drop_duplicates(
            subset=["GAME_ID"]
        )

        raw.to_csv(raw_path, index=False)

        if failed_dates:
            failure_path = (
                RAW_DIR / f"failed_dates_{season}.csv"
            )
            pd.DataFrame({
                "DATE": failed_dates
            }).to_csv(failure_path, index=False)

            print("Failed dates:", len(failed_dates))

    clean = clean_games(raw, season)
    counts = validate(clean, season)

    clean.to_csv(clean_path, index=False)

    all_clean.append(clean)

    expected = EXPECTED[season]

    valid_count = (
        expected is not None
        and len(clean) == expected
    )

    validation_results.append({
        "SEASON": season,
        "GAMES": len(clean),
        "TEAMS": len(counts),
        "EXPECTED": expected,
        "COUNT_MATCH": valid_count
    })

    print("Saved:", clean_path)


# ==========================================
# COMBINE ALL SEASONS
# ==========================================

if all_clean:
    combined = pd.concat(
        all_clean, ignore_index=True
    )

    combined = combined.drop_duplicates(
        subset=["SEASON", "GAME_ID"]
    )

    combined_path = (
        BASE_DIR / "nba_games_2015_2025_combined.csv"
    )

    combined.to_csv(
        combined_path, index=False
    )

    print("\nCombined dataset saved:")
    print(combined_path)
    print("Total games:", len(combined))

validation_df = pd.DataFrame(validation_results)

validation_path = BASE_DIR / "validation_report.csv"
validation_df.to_csv(validation_path, index=False)

print("\nVALIDATION REPORT")
print(validation_df.to_string(index=False))


Processing: 2015-16


2015-16:   0%|          | 0/213 [00:00<?, ?it/s]


Season: 2015-16
Total games: 1230
Unique teams: 30
Expected: 1230
Saved: /content/drive/MyDrive/Colab Notebooks/Dataset/NBA Dataset/clean/nba_games_2015-16_clean.csv

Processing: 2016-17


2016-17:   0%|          | 0/212 [00:00<?, ?it/s]


Season: 2016-17
Total games: 1230
Unique teams: 30
Expected: 1230
Saved: /content/drive/MyDrive/Colab Notebooks/Dataset/NBA Dataset/clean/nba_games_2016-17_clean.csv

Processing: 2017-18


...


Season: 2022-23
Total games: 1230
Unique teams: 30
Expected: 1230
Saved: /content/drive/MyDrive/Colab Notebooks/Dataset/NBA Dataset/clean/nba_games_2022-23_clean.csv

Processing: 2023-24


2023-24:   0%|          | 0/213 [00:00<?, ?it/s]


Season: 2023-24
Total games: 1230
Unique teams: 30
Expected: 1230
Saved: /content/drive/MyDrive/Colab Notebooks/Dataset/NBA Dataset/clean/nba_games_2023-24_clean.csv

Processing: 2024-25


2024-25:   0%|          | 0/212 [00:00<?, ?it/s]


Season: 2024-25
Total games: 1230
Unique teams: 30
Expected: 1230
Saved: /content/drive/MyDrive/Colab Notebooks/Dataset/NBA Dataset/clean/nba_games_2024-25_clean.csv

Combined dataset saved:
/content/drive/MyDrive/Colab Notebooks/Dataset/NBA Dataset/nba_games_2015_2025_combined.csv
Total games: 11979

VALIDATION REPORT
 SEASON  GAMES  TEAMS  EXPECTED  COUNT_MATCH
2015-16   1230     30    1230.0         True
2016-17   1230     30    1230.0         True
2017-18   1230     30    1230.0         True
2018-19   1230     30    1230.0         True
2019-20   1059     30       NaN        False
2020-21   1080     30    1080.0         True
2021-22   1230     30    1230.0         True
2022-23   1230     30    1230.0         True
2023-24   1230     30    1230.0         True
2024-25   1230     30    1230.0         True


In [14]:
import pandas as pd

path = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "Dataset/NBA Dataset/validation_report.csv"
)

report = pd.read_csv(path)

report.loc[
    report["SEASON"] == "2019-20", "EXPECTED"
] = 1059

report["COUNT_MATCH"] = (
    report["GAMES"] == report["EXPECTED"]
)

report.to_csv(path, index=False)

print(report.to_string(index=False))
print("\nAll seasons valid:", report["COUNT_MATCH"].all())

 SEASON  GAMES  TEAMS  EXPECTED  COUNT_MATCH
2015-16   1230     30    1230.0         True
2016-17   1230     30    1230.0         True
2017-18   1230     30    1230.0         True
2018-19   1230     30    1230.0         True
2019-20   1059     30    1059.0         True
2020-21   1080     30    1080.0         True
2021-22   1230     30    1230.0         True
2022-23   1230     30    1230.0         True
2023-24   1230     30    1230.0         True
2024-25   1230     30    1230.0         True

All seasons valid: True


In [15]:

import pandas as pd
import requests
import time
from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Dataset/NBA Dataset"
)

games = pd.read_csv(
    BASE_DIR / "nba_games_2015_2025_combined.csv",
    dtype={"GAME_ID": str}
)

URL = (
    "https://site.api.espn.com/apis/site/v2/"
    "sports/basketball/nba/summary"
)

session = requests.Session()

def get_player_boxscore(game):
    game_id = str(game["GAME_ID"])

    response = session.get(
        URL,
        params={"event": game_id},
        timeout=30
    )
    response.raise_for_status()
    data = response.json()

    rows = []

    for team_block in data.get("boxscore", {}).get("players", []):
        team = team_block.get("team", {})

        for stat_group in team_block.get("statistics", []):
            names = stat_group.get("names", [])

            for entry in stat_group.get("athletes", []):
                athlete = entry.get("athlete", {})
                values = entry.get("stats", [])

                # ESPN's stat labels and values are parallel arrays
                stats = dict(zip(names, values))

                row = {
                    "SEASON": game["SEASON"],
                    "GAME_ID": game_id,
                    "GAME_DATE": game["GAME_DATE"],
                    "TEAM_ID": str(team.get("id", "")),
                    "TEAM": team.get("displayName"),
                    "PLAYER_ID": str(athlete.get("id", "")),
                    "PLAYER": athlete.get("displayName"),
                    "POSITION": athlete.get(
                        "position", {}
                    ).get("abbreviation"),
                    "STARTER": entry.get("starter"),
                    "DID_NOT_PLAY": entry.get("didNotPlay"),
                }

                row.update(stats)
                rows.append(row)

    return rows


# Test five games from different seasons
sample_games = (
    games.groupby("SEASON", group_keys=False)
    .head(1)
    .head(5)
)

records = []

for _, game in sample_games.iterrows():
    try:
        result = get_player_boxscore(game)
        records.extend(result)

        print(
            game["SEASON"],
            game["GAME_ID"],
            "Player records:", len(result)
        )

    except Exception as e:
        print("Error:", game["GAME_ID"], e)

    time.sleep(1.2)

player_test = pd.DataFrame(records)

print("\nTotal player records:", len(player_test))

if not player_test.empty:
    print("\nColumns:", player_test.columns.tolist())
    display(player_test.head(15))

    player_test.to_csv(
        BASE_DIR / "player_boxscore_test.csv",
        index=False
    )


2015-16 400827888 Player records: 26
2016-17 400899375 Player records: 26
2017-18 400974437 Player records: 25
2018-19 401070213 Player records: 26
2019-20 401160623 Player records: 26

Total player records: 129

Columns: ['SEASON', 'GAME_ID', 'GAME_DATE', 'TEAM_ID', 'TEAM', 'PLAYER_ID', 'PLAYER', 'POSITION', 'STARTER', 'DID_NOT_PLAY', 'MIN', 'PTS', 'FG', '3PT', 'FT', 'REB', 'AST', 'TO', 'STL', 'BLK', 'OREB', 'DREB', 'PF', '+/-']


,SEASON,GAME_ID,GAME_DATE,TEAM_ID,TEAM,PLAYER_ID,PLAYER,POSITION,STARTER,DID_NOT_PLAY,...,FT,REB,AST,TO,STL,BLK,OREB,DREB,PF,+/-
0,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,2767,Ersan Ilyasova,PF,True,False,...,1-2,7,3,3,0,1,3,4,4,+20
1,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,6462,Marcus Morris Sr.,F,True,False,...,5-6,10,4,0,0,0,5,5,1,+17
2,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,6585,Andre Drummond,C,True,False,...,6-10,19,3,2,1,2,8,11,4,+23
3,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,6443,Reggie Jackson,PG,True,False,...,5-5,8,5,2,2,0,1,7,0,+26
4,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,2581018,Kentavious Caldwell-Pope,G,True,False,...,3-3,4,1,2,1,0,1,3,1,+17
5,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,3134881,Stanley Johnson,SF,False,False,...,0-0,4,3,1,0,0,3,1,1,-13
6,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,2968439,Aron Baynes,C,False,False,...,0-0,5,0,1,0,0,1,4,2,-11
7,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,1994,Steve Blake,PG,False,False,...,0-0,0,4,3,1,0,0,0,1,-14
8,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,4003,Jodie Meeks,G,False,False,...,0-0,2,0,1,0,0,1,1,1,-5
9,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,2528779,Reggie Bullock Jr.,SF,False,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
print("Available columns:")
print(player_test.columns.tolist())

print("\nStarter values:")
print(player_test["STARTER"].value_counts(dropna=False))

print("\nSample player statistics:")
display(player_test.head(10))

Available columns:
['SEASON', 'GAME_ID', 'GAME_DATE', 'TEAM_ID', 'TEAM', 'PLAYER_ID', 'PLAYER', 'POSITION', 'STARTER', 'DID_NOT_PLAY', 'MIN', 'PTS', 'FG', '3PT', 'FT', 'REB', 'AST', 'TO', 'STL', 'BLK', 'OREB', 'DREB', 'PF', '+/-']

Starter values:
STARTER
False    79
True     50
Name: count, dtype: int64

Sample player statistics:


,SEASON,GAME_ID,GAME_DATE,TEAM_ID,TEAM,PLAYER_ID,PLAYER,POSITION,STARTER,DID_NOT_PLAY,...,FT,REB,AST,TO,STL,BLK,OREB,DREB,PF,+/-
0,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,2767,Ersan Ilyasova,PF,True,False,...,1-2,7,3,3,0,1,3,4,4,+20
1,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,6462,Marcus Morris Sr.,F,True,False,...,5-6,10,4,0,0,0,5,5,1,+17
2,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,6585,Andre Drummond,C,True,False,...,6-10,19,3,2,1,2,8,11,4,+23
3,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,6443,Reggie Jackson,PG,True,False,...,5-5,8,5,2,2,0,1,7,0,+26
4,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,2581018,Kentavious Caldwell-Pope,G,True,False,...,3-3,4,1,2,1,0,1,3,1,+17
5,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,3134881,Stanley Johnson,SF,False,False,...,0-0,4,3,1,0,0,3,1,1,-13
6,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,2968439,Aron Baynes,C,False,False,...,0-0,5,0,1,0,0,1,4,2,-11
7,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,1994,Steve Blake,PG,False,False,...,0-0,0,4,3,1,0,0,0,1,-14
8,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,4003,Jodie Meeks,G,False,False,...,0-0,2,0,1,0,0,1,1,1,-5
9,2015-16,400827888,2015-10-28T00:00Z,8,Detroit Pistons,2528779,Reggie Bullock Jr.,SF,False,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
starter_check = (
    player_test.groupby(
        ["SEASON", "GAME_ID", "TEAM"]
    )["STARTER"]
    .sum()
    .reset_index(name="STARTER_COUNT")
)

display(starter_check)

print(
    "All teams have 5 starters:",
    starter_check["STARTER_COUNT"].eq(5).all()
)

,SEASON,GAME_ID,TEAM,STARTER_COUNT
0,2015-16,400827888,Atlanta Hawks,5
1,2015-16,400827888,Detroit Pistons,5
2,2016-17,400899375,Cleveland Cavaliers,5
3,2016-17,400899375,New York Knicks,5
4,2017-18,400974437,Boston Celtics,5
5,2017-18,400974437,Cleveland Cavaliers,5
6,2018-19,401070213,Boston Celtics,5
7,2018-19,401070213,Philadelphia 76ers,5
8,2019-20,401160623,New Orleans Pelicans,5
9,2019-20,401160623,Toronto Raptors,5


All teams have 5 starters: True


In [19]:
import pandas as pd
import requests
import time
from pathlib import Path
from tqdm.auto import tqdm

BASE_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/Dataset/NBA Dataset"
)

OUTPUT_DIR = BASE_DIR / "player_boxscores"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GAMES_FILE = BASE_DIR / "nba_games_2015_2025_combined.csv"

BATCH_SIZE = 100
REQUEST_DELAY = 0.8
MAX_RETRIES = 3

API_URL = (
    "https://site.api.espn.com/apis/site/v2/"
    "sports/basketball/nba/summary"
)

games = pd.read_csv(
    GAMES_FILE,
    dtype={"GAME_ID": str}
)

games = (
    games.drop_duplicates(subset=["SEASON", "GAME_ID"])
    .sort_values(["SEASON", "GAME_DATE", "GAME_ID"])
    .reset_index(drop=True)
)

session = requests.Session()


def get_player_boxscore(game):
    game_id = str(game["GAME_ID"])

    response = session.get(
        API_URL,
        params={"event": game_id},
        timeout=30
    )

    response.raise_for_status()
    data = response.json()

    rows = []

    for team_block in data.get("boxscore", {}).get("players", []):
        team = team_block.get("team", {})

        for stat_group in team_block.get("statistics", []):
            stat_names = stat_group.get("names", [])

            for entry in stat_group.get("athletes", []):
                athlete = entry.get("athlete", {})
                stat_values = entry.get("stats", [])

                stats = dict(zip(stat_names, stat_values))

                position = athlete.get("position") or {}

                row = {
                    "SEASON": game["SEASON"],
                    "GAME_ID": game_id,
                    "GAME_DATE": game["GAME_DATE"],
                    "TEAM_ID": str(team.get("id", "")),
                    "TEAM": team.get("displayName"),
                    "PLAYER_ID": str(athlete.get("id", "")),
                    "PLAYER": athlete.get("displayName"),
                    "POSITION": position.get("abbreviation"),
                    "STARTER": entry.get("starter", False),
                    "DID_NOT_PLAY": entry.get("didNotPlay", False)
                }

                row.update(stats)
                rows.append(row)

    if not rows:
        raise ValueError("Empty player box score")

    return rows


def download_with_retry(game):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return get_player_boxscore(game)

        except (
            requests.RequestException,
            ValueError,
            KeyError,
            TypeError
        ) as error:

            print(
                f"Game {game['GAME_ID']} | "
                f"Attempt {attempt}/{MAX_RETRIES} | "
                f"{error}"
            )

            if attempt < MAX_RETRIES:
                time.sleep(3 * attempt)

    return None


total_batches = (
    len(games) + BATCH_SIZE - 1
) // BATCH_SIZE

print("Total games:", len(games))
print("Total batches:", total_batches)

for batch_num in tqdm(
    range(total_batches),
    desc="Downloading NBA player box scores"
):
    start = batch_num * BATCH_SIZE
    end = min(start + BATCH_SIZE, len(games))

    batch_games = games.iloc[start:end]

    batch_path = (
        OUTPUT_DIR / f"players_batch_{batch_num:04d}.csv"
    )

    failure_path = (
        OUTPUT_DIR / f"failed_batch_{batch_num:04d}.csv"
    )

    if batch_path.exists():
        batch_df = pd.read_csv(
            batch_path,
            dtype={"GAME_ID": str, "PLAYER_ID": str}
        )

        records = batch_df.to_dict("records")
        completed_ids = set(
            batch_df["GAME_ID"].astype(str)
        )

    else:
        records = []
        completed_ids = set()

    for _, game in batch_games.iterrows():
        game_id = str(game["GAME_ID"])

        if game_id in completed_ids:
            continue

        result = download_with_retry(game)

        if result is not None:
            records.extend(result)
            completed_ids.add(game_id)

        time.sleep(REQUEST_DELAY)

    if records:
        batch_df = pd.DataFrame(records)

        batch_df = batch_df.drop_duplicates(
            subset=["GAME_ID", "TEAM_ID", "PLAYER_ID"]
        )

        batch_df.to_csv(
            batch_path,
            index=False
        )

    missing_games = batch_games[
        ~batch_games["GAME_ID"].astype(str).isin(completed_ids)
    ][["SEASON", "GAME_ID"]]

    if not missing_games.empty:
        missing_games.to_csv(
            failure_path,
            index=False
        )

    elif failure_path.exists():
        failure_path.unlink()

    print(
        f"Batch {batch_num + 1}/{total_batches} | "
        f"Completed games: {len(completed_ids)}/{len(batch_games)} | "
        f"Failed games: {len(missing_games)}"
    )


batch_files = sorted(
    OUTPUT_DIR.glob("players_batch_*.csv")
)

if not batch_files:
    raise RuntimeError("No player box-score files were downloaded.")

player_data = pd.concat(
    [
        pd.read_csv(
            file,
            dtype={
                "GAME_ID": str,
                "PLAYER_ID": str,
                "TEAM_ID": str
            }
        )
        for file in batch_files
    ],
    ignore_index=True
)

player_data = player_data.drop_duplicates(
    subset=["GAME_ID", "TEAM_ID", "PLAYER_ID"]
)

player_data = player_data.sort_values(
    ["SEASON", "GAME_DATE", "GAME_ID", "TEAM"]
).reset_index(drop=True)

OUTPUT_FILE = (
    BASE_DIR / "nba_player_boxscores_2015_2025.csv"
)

player_data.to_csv(
    OUTPUT_FILE,
    index=False
)

expected_ids = set(games["GAME_ID"].astype(str))
downloaded_ids = set(player_data["GAME_ID"].astype(str))

missing_ids = expected_ids - downloaded_ids

print("\nDOWNLOAD SUMMARY")
print("=" * 50)

print("Expected games:", len(expected_ids))
print("Downloaded games:", len(downloaded_ids))
print("Missing games:", len(missing_ids))
print("Total player records:", len(player_data))
print("Unique players:", player_data["PLAYER_ID"].nunique())

print("\nPlayers by season:")

season_summary = (
    player_data.groupby("SEASON")
    .agg(
        GAMES=("GAME_ID", "nunique"),
        PLAYERS=("PLAYER_ID", "nunique"),
        PLAYER_RECORDS=("PLAYER_ID", "size")
    )
    .reset_index()
)

display(season_summary)

if missing_ids:
    missing_report = games[
        games["GAME_ID"].astype(str).isin(missing_ids)
    ]

    missing_report.to_csv(
        OUTPUT_DIR / "missing_games.csv",
        index=False
    )

    print("\nMissing games saved to missing_games.csv")

else:
    print("\nAll games have player box-score records.")

print("\nDataset saved to:")
print(OUTPUT_FILE)

Total games: 11979
Total batches: 120


Batch 1/120 | Completed games: 100/100 | Failed games: 0
Batch 2/120 | Completed games: 100/100 | Failed games: 0
Batch 3/120 | Completed games: 100/100 | Failed games: 0
Batch 4/120 | Completed games: 100/100 | Failed games: 0
Batch 5/120 | Completed games: 100/100 | Failed games: 0
Batch 6/120 | Completed games: 100/100 | Failed games: 0
Batch 7/120 | Completed games: 100/100 | Failed games: 0
Batch 8/120 | Completed games: 100/100 | Failed games: 0
Batch 9/120 | Completed games: 100/100 | Failed games: 0
Batch 10/120 | Completed games: 100/100 | Failed games: 0
Batch 11/120 | Completed games: 100/100 | Failed games: 0
Batch 12/120 | Completed games: 100/100 | Failed games: 0
Batch 13/120 | Completed games: 100/100 | Failed games: 0
Batch 14/120 | Completed games: 100/100 | Failed games: 0
Batch 15/120 | Completed games: 100/100 | Failed games: 0
Batch 16/120 | Completed games: 100/100 | Failed games: 0
Batch 17/120 | Completed games: 100/100 | Failed games: 0
Batch 18/120 | Complete

,SEASON,GAMES,PLAYERS,PLAYER_RECORDS
0,2015-16,1230,478,31454
1,2016-17,1230,490,31585
2,2017-18,1230,550,30990
3,2018-19,1230,545,30843
4,2019-20,1059,537,26544
5,2020-21,1080,544,28884
6,2021-22,1230,622,31330
7,2022-23,1230,548,31551
8,2023-24,1230,588,32500
9,2024-25,1230,580,32582



All games have player box-score records.

Dataset saved to:
/content/drive/MyDrive/Colab Notebooks/Dataset/NBA Dataset/nba_player_boxscores_2015_2025.csv


In [20]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/Dataset/NBA Dataset"
)

games = pd.read_csv(
    BASE_DIR / "nba_games_2015_2025_combined.csv",
    dtype={"GAME_ID": str}
)

players = pd.read_csv(
    BASE_DIR / "nba_player_boxscores_2015_2025.csv",
    dtype={"GAME_ID": str, "PLAYER_ID": str, "TEAM_ID": str}
)

print("Games:", games.shape)
print("Players:", players.shape)

print("\nGame columns:")
print(games.columns.tolist())

print("\nPlayer columns:")
print(players.columns.tolist())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Games: (11979, 11)
Players: (308263, 24)

Game columns:
['SEASON', 'GAME_ID', 'GAME_DATE', 'HOME_TEAM_ID', 'HOME_TEAM', 'AWAY_TEAM_ID', 'AWAY_TEAM', 'HOME_SCORE', 'AWAY_SCORE', 'HOME_WIN', 'EVENT_NAME']

Player columns:
['SEASON', 'GAME_ID', 'GAME_DATE', 'TEAM_ID', 'TEAM', 'PLAYER_ID', 'PLAYER', 'POSITION', 'STARTER', 'DID_NOT_PLAY', 'MIN', 'PTS', 'FG', '3PT', 'FT', 'REB', 'AST', 'TO', 'STL', 'BLK', 'OREB', 'DREB', 'PF', '+/-']


In [21]:
# Check game IDs are unique
assert not games["GAME_ID"].duplicated().any(), \
    "Duplicate GAME_ID values in games dataset"

# Check every player record matches a game
missing_ids = set(players["GAME_ID"]) - set(games["GAME_ID"])
print("Player game IDs missing from games:", len(missing_ids))

# Combine player statistics with game information
combined = players.merge(
    games.drop(columns=["SEASON", "GAME_DATE"], errors="ignore"),
    on="GAME_ID",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("\nMerge results:")
print(combined["_merge"].value_counts())

# Check for unmatched records
assert (combined["_merge"] == "both").all(), \
    "Some player records did not match a game"

combined = combined.drop(columns="_merge")

print("\nCombined dataset:", combined.shape)
display(combined.head())

Player game IDs missing from games: 0

Merge results:
_merge
both          308263
left_only          0
right_only         0
Name: count, dtype: int64

Combined dataset: (308263, 32)


,SEASON,GAME_ID,GAME_DATE,TEAM_ID,TEAM,PLAYER_ID,PLAYER,POSITION,STARTER,DID_NOT_PLAY,...,PF,+/-,HOME_TEAM_ID,HOME_TEAM,AWAY_TEAM_ID,AWAY_TEAM,HOME_SCORE,AWAY_SCORE,HOME_WIN,EVENT_NAME
0,2015-16,400827888,2015-10-28T00:00Z,1,Atlanta Hawks,3015,Paul Millsap,PF,True,False,...,4.0,-22.0,1,Atlanta Hawks,8,Detroit Pistons,94,106,0,Detroit Pistons at Atlanta Hawks
1,2015-16,400827888,2015-10-28T00:00Z,1,Atlanta Hawks,6637,Kent Bazemore,SF,True,False,...,3.0,-17.0,1,Atlanta Hawks,8,Detroit Pistons,94,106,0,Detroit Pistons at Atlanta Hawks
2,2015-16,400827888,2015-10-28T00:00Z,1,Atlanta Hawks,3213,Al Horford,C,True,False,...,1.0,-5.0,1,Atlanta Hawks,8,Detroit Pistons,94,106,0,Detroit Pistons at Atlanta Hawks
3,2015-16,400827888,2015-10-28T00:00Z,1,Atlanta Hawks,4015,Jeff Teague,PG,True,False,...,1.0,-23.0,1,Atlanta Hawks,8,Detroit Pistons,94,106,0,Detroit Pistons at Atlanta Hawks
4,2015-16,400827888,2015-10-28T00:00Z,1,Atlanta Hawks,2011,Kyle Korver,G,True,False,...,4.0,-9.0,1,Atlanta Hawks,8,Detroit Pistons,94,106,0,Detroit Pistons at Atlanta Hawks


In [22]:
output_path = BASE_DIR / "nba_complete_dataset_2015_2025.csv"

combined.to_csv(output_path, index=False)

print("Dataset saved successfully!")
print("Location:", output_path)
print("Rows:", len(combined))
print("Columns:", len(combined.columns))

Dataset saved successfully!
Location: /content/drive/MyDrive/Colab Notebooks/Dataset/NBA Dataset/nba_complete_dataset_2015_2025.csv
Rows: 308263
Columns: 32
